In [ ]:
import random, time, numpy as np, torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.FashionMNIST(

root='data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(
    root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)


100%|██████████| 26.4M/26.4M [00:01<00:00, 14.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 236kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 4.33MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.8MB/s]


In [ ]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # STUDENT: define three trainable Linear layers
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784,128)
        self.fc2 = nn.Linear(128,64)
        self.fc3 = nn.Linear(64,10)
        self.activation = nn.ReLU()

    def forward(self, x):
        # STUDENT: flatten, apply Layer 1 + activation,
        # Layer 2 + activation, then Layer 3 logits
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        logits = self.fc3(x)

        return logits

model = FashionMLP().to(device)
print(model)


FashionMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
  (activation): ReLU()
)


Layer 1 uses 784 inputs because the picture has 28x28 pixels, which is a 2 dimensional matrix and it needs to be flattened into a one-dimensional vector. Since it has 28x28 pixels, the total is 784, so the first layer requires 784 input neurons. ReLU was use because this helps the model focus on importamt signals and makes training faster. Layer 3 has 10 outputs because layer 3 receives 65 inputs from the previous hidden layer and produces 10 output (nn.Linear(64,10)), that the model uses to make its final prediction. Logits are basically raw scores, and when the model returns logits, the loss function will use them to check how wrong the prediction is by comparing it with the correct answer.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch(model, loader):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # clear previous gradients
        logits = model(images)         # forward propagation
        loss = criterion(logits, labels)
        loss.backward()                # backpropagation
        optimizer.step()               # parameter update

        running_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total
